# **TUTORIAL** Study objective A (uncertainty)


AESA Phase A: estimating the environmental burden.\
Compute IO-LCA outputs from processed MRIO tables and run IO-LCA Monte Carlo
**uncertainty** from deterministic IO-LCA outputs. 

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommend to first go through all notebooks available in the `tutorials/core_prerequisites` folder, to have extensive details on what the functions do.

### Functional units and selectors

The example in this tutorial use the **functional unit** `L2.c.b` with the following **selectors**: one producing sector `s_p="Paper"` and one consuming region `r_c="FR"`.

For more details on functional units, please check out:
- the short guide to functional units, region/sector selectors and allocation methods at `tutorials/study_objectives/1_functional_units_and_allocation_methods.md` for a streamlined explanation.
- the dedicated complete documentation in the methodological notes at `methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf` for an in-depth explanation.

In [ ]:
project_name_ = "SO_A_demo_paper_fr"

source_ = "exiobase_3102_ixi"
fu_code_ = "L2.c.b"
study_period_ = range(2010,2021)
s_p_ = ["Paper"]
r_c_ = ["FR"]
lcia_method_ = "pb_lcia"

# Basic features of the uncertainty function

### In a nutshell

The function takes necessary **inputs**, for two main purposes:
- inputs for the deterministic function with the `base_io_lca_args` block, which mostly includes:
    - `project_name`, `source`, `years`, `lcia_method`, `fu_code` and the relevant selectors (here `s_p` and `r_c`) for the selected functional unit.
- inputs for the configuration of uncertainty analysis (Monte Carlo) through the `uncertainty_config` block, which mostly includes:
    - `mc_parameters` and `lcia_uncertainty` blocks, with nested parameters.

The uncertainty **output** folder contains:
- Monte Carlo run values and summary statistics.
- **Figures** are rendered when requested, as controlled with the `figures` and `figure_format` parameters.
- log files, including uncertainty source parameter logs.

Basic features also involve:
- **grouping** of sectors and/or regions: use the `group_sec`, `group_reg`, and `group_version` parameters.
- **overwriting** of existing values: use the `refresh` parameter.

The uncertainty source note is
`data_raw/methodological_notes/methodological_note__acc_uncertainty_sources.pdf`.

 

### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>uncertainty_io_lca(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>base_io_lca_args</code></td><td>Deterministic IO-LCA selector envelope.<br>&#10;Write nested arguments as <code>base_io_lca_args={&quot;project_name&quot;: &quot;...&quot;, &quot;source&quot;: &quot;...&quot;, &quot;lcia_method&quot;: &quot;...&quot;, &quot;fu_code&quot;: &quot;...&quot;}</code>. Required keys are <code>project_name</code>, <code>source</code>,<br>&#10;<code>lcia_method</code>, and <code>fu_code</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>project_name</code>: Required project name used to build<br>&#10;  <code>&lt;repo&gt;/&lt;project_name&gt;</code>.<br>&#10;&bull; <code>source</code>: MRIO source key (<code>&quot;exiobase_396_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_396_pxp&quot;</code>, <code>&quot;exiobase_3102_ixi&quot;</code>,<br>&#10;  <code>&quot;exiobase_3102_pxp&quot;</code>). pyaesa currently only supports<br>&#10;  EXIOBASE for LCIA characterization.<br>&#10;<span style="color:#c45f00">&bull; <code>group_reg</code>: If <code>True</code>, aggregate regions using a grouping<br>&#10;  file. <strong>Default</strong> <code>False</code> keeps native source regions.</span><br>&#10;<span style="color:#c45f00">&bull; <code>group_sec</code>: If <code>True</code>, aggregate sectors using a grouping<br>&#10;  file. <strong>Default</strong> <code>False</code> keeps native source sectors.</span><br>&#10;<span style="color:#c45f00">&bull; <code>group_version</code>: Grouping version tag used to resolve the<br>&#10;  region/sector mapping CSVs. Required when <code>group_reg</code> or<br>&#10;  <code>group_sec</code> is True. <strong>Defaults to</strong> an empty string for ungrouped<br>&#10;  processing. Follow <code>README_grouping.txt</code> in the active<br>&#10;  <code>data_raw/mrio/&lt;source&gt;/grouping</code> folder to name grouping<br>&#10;  versions and place the matching mapping CSVs.</span><br>&#10;&bull; <code>years</code>: Studied years. Accepts a single year, list, or<br>&#10;  range. <strong>If omitted</strong>, all available MRIO years for the selected<br>&#10;  source/group version are used.<br>&#10;&bull; <code>lcia_method</code>: Required LCIA method(s) selected for IO-LCA<br>&#10;  results (for example <code>&quot;pb_lcia&quot;</code> or<br>&#10;  <code>[&quot;pb_lcia&quot;, &quot;gwp100_lcia&quot;]</code>). The method(s) must have been<br>&#10;  processed for the same MRIO source with <code>process_mrio(...)</code>.<br>&#10;  pyaesa currently supports IO-LCA only for EXIOBASE sources. To<br>&#10;  add a custom LCIA method with which run <code>process_mrio(...)</code>,<br>&#10;  follow<br>&#10;  <code>README_add_custom_lcia_characterization_matrices.txt</code> in<br>&#10;  <code>data_raw/mrio/exiobase_3/lcia/characterization_factors_matrices/</code><br>&#10;  and pass the custom method file stem here.<br>&#10;&bull; <code>fu_code</code>: Required functional unit code (for example<br>&#10;  <code>&quot;L1.a&quot;</code>, <code>&quot;L2.c.b&quot;</code>). See<br>&#10;  <code>data_raw/methodological_notes/methodological_note__asocc_fus_allocation_methods.pdf</code><br>&#10;  for all available functional unit codes and the system<br>&#10;  boundaries each represents.<br>&#10;&bull; <code>s_p</code>: Producing sector filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing sectors. To<br>&#10;  identify valid sector names, see the first column of the<br>&#10;  relevant<br>&#10;  <code>data_raw/mrio/.../grouping/.../group_sec_template.csv</code> file.<br>&#10;  For EXIOBASE sector definitions, see<br>&#10;  <code>data_raw/mrio/exiobase_3/sector_classification.xlsx</code>;<br>&#10;  EXIOBASE ixi and pxp use different sector lists.<br>&#10;&bull; <code>r_p</code>: Producing region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid producing regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/group_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull; <code>r_c</code>: Consuming region filter(s), single string or list. If<br>&#10;  this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid consuming regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/group_reg_template.csv</code><br>&#10;  file.<br>&#10;&bull; <code>r_f</code>: Final demand region filter(s), single string or list.<br>&#10;  If this is a required axis for <code>fu_code</code> and the argument is<br>&#10;  <strong>omitted</strong>, the run expands to all valid final demand regions. To<br>&#10;  identify valid region names, see the first column of the<br>&#10;  relevant <code>data_raw/mrio/.../grouping/group_reg_template.csv</code><br>&#10;  file.<br>&#10;<span style="color:#087f5b">&bull; <code>aggreg_indices</code>: Whether multiple selected region/sector<br>&#10;  indices are reported as separate rows or summed into one row<br>&#10;  after the selected MRIO scope is computed.<br>&#10;  &bull; <code>False</code> (<strong>default</strong>): keep selected values as independent rows.<br>&#10;  &bull; <code>True</code>: sum selected values into one row.<br>&#10;  Not allowed for <code>L2.a.b</code>, <code>L2.b.b</code>, and <code>L2.c.b</code> because<br>&#10;  aggregating CBA total demand system boundaries can double count.<br>&#10;  For these functional units, define the aggregation from<br>&#10;  <code>process_mrio(...)</code> onward with<br>&#10;  <code>group_reg</code>/<code>group_sec</code>/<code>group_version</code>.</span></td></tr>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>uncertainty_config</code></td><td>Monte Carlo configuration dictionary. It must<br>&#10;activate <code>lcia_uncertainty</code>, which is the IO-LCA public<br>&#10;uncertainty source. LCIA uncertainty is inactive by <strong>default</strong><br>&#10;because L2 LCIA rows require <code>sector_cov_mapping</code>:<br>&#10;keys are output <code>s_p</code> labels and values are sector CoV codes from<br>&#10;<code>sec_cbca_covs.csv</code>. Source blocks use an <code>active</code> boolean. See<br>&#10;<code>data_raw/methodological_notes/methodological_note__acc_uncertainty_sources.pdf</code><br>&#10;for uncertainty source definitions and mathematical expressions.<br>&#10;&#160;<br>&#10;Accepted keys:<br>&#10;&#160;<br>&#10;<span style="color:#087f5b">&bull; <code>mc_parameters</code>: optional dictionary with <code>convergence</code> and<br>&#10;  <code>fixed</code> mode blocks. Exactly one mode must be active.<br>&#10;&#160;<br>&#10;  Nested mode blocks:<br>&#10;</span><br>&#10;<span style="color:#087f5b">  &bull; <code>convergence</code>: convergence mode block. This is the <strong>default</strong><br>&#10;    active mode.<br>&#10;&#160;<br>&#10;    Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">    &bull; <code>active</code>: Whether convergence mode is active.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>max_runs</code>: Maximum number of Monte Carlo runs allowed<br>&#10;      before stopping.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>rtol</code>: Relative tolerance used to decide whether<br>&#10;      monitored summary statistics have converged.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>stable_runs</code>: Number of consecutive accepted runs that<br>&#10;      must remain within tolerance before the run stops.</span><br>&#10;<span style="color:#087f5b">    &bull; <code>convergence_statistics</code>: Statistic monitored for<br>&#10;      convergence. Monte Carlo convergence is mean only and<br>&#10;      <strong>defaults to</strong> <code>[&quot;mean&quot;]</code>.</span><br>&#10;&#160;<br>&#10;<span style="color:#c45f00">  &bull; <code>fixed</code>: fixed run count mode block.<br>&#10;&#160;<br>&#10;    Nested keys:<br>&#10;</span><br>&#10;<span style="color:#087f5b">    &bull; <code>active</code>: Whether fixed mode is active.</span><br>&#10;    &bull; <code>n_runs</code>: Exact number of Monte Carlo runs.<br>&#10;&#160;<br>&#10;<span style="color:#c45f00">&bull; <code>lcia_uncertainty</code>: optional LCIA source block. It <strong>defaults<br>&#10;  to</strong> <code>{&quot;active&quot;: False, &quot;sector_cov_mapping&quot;: {}}</code>. Country<br>&#10;  level LCIA CoVs are resolved automatically. L2 sector resolved<br>&#10;  LCIA rows require <code>sector_cov_mapping</code> to map output <code>s_p</code><br>&#10;  labels to sector CoV codes from <code>sec_cbca_covs.csv</code>, for example<br>&#10;  <code>{&quot;active&quot;: True, &quot;sector_cov_mapping&quot;: {&quot;Paper&quot;: &quot;Paper&quot;}}</code>.<br>&#10;  Carbon consumption based accounts<br>&#10;  coefficients of variation (CoV) files are available under<br>&#10;  <code>data_raw/mrio/exiobase_3/lcia/carbon_accounts_covs/</code>. Users<br>&#10;  can inspect <code>sec_cbca_covs.csv</code> for sector CoV codes before<br>&#10;  choosing <code>sector_cov_mapping</code> values.<br>&#10;&#160;<br>&#10;  Nested keys:<br>&#10;</span><br>&#10;<span style="color:#c45f00">  &bull; <code>active</code>: Whether LCIA uncertainty is active.</span><br>&#10;<span style="color:#c45f00">  &bull; <code>sector_cov_mapping</code>: Mapping from output <code>s_p</code></span><br>&#10;    labels to sector CoV codes from <code>sec_cbca_covs.csv</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Public uncertainty table format, either<br>&#10;<code>&quot;csv_compact&quot;</code> or <code>&quot;parquet&quot;</code>. <strong>Defaults to</strong><br>&#10;<code>&quot;csv_compact&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull; <code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, refresh both the resolved deterministic IO-LCA prerequisite and the resolved IO-LCA Monte Carlo outputs for this uncertainty request. The deterministic refresh clears the selected <code>deterministic_io_lca(...)</code> source and version output scope under <code>&lt;project&gt;/A_lca/io_lca</code>. The Monte Carlo refresh removes matching run folders for the current request under the adjacent <code>monte_carlo</code> root. Processed MRIO inputs, processed population and GDP, raw downloads, and downstream ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>



### Uncertainty IO-LCA with defaults where omitted


In [ ]:
from pyaesa import uncertainty_io_lca

uncertainty_io_lca(
    base_io_lca_args={
        "project_name": project_name_,
        "source": source_,
        "years": study_period_,
        "lcia_method": lcia_method_,
        "fu_code": fu_code_,
        "s_p": s_p_,
        "r_c": r_c_,
    },
    uncertainty_config={
        "lcia_uncertainty": {
            "active": True,
            "sector_cov_mapping": {"Paper": "Paper"},
        },
    },
    refresh=True
)

### External LCA results

If you want to use **externally provided LCA results** instead of <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span> IO-LCA results computed with EXIOBASE for the ASR numerator, check out the tutorial at `tutorials\optional_workflows\external_asocc_lca_input_staging.ipynb`


# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at `tutorials/study_objectives/...`

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available includes:
- custom definition of CoVs values for grouped countries

<!-- - (move to uncertainty) inter-mrio uncertainty
- (asocc) custom weighting between allocation methods
- disaggragation
- (asocc) external allocation method -->

### Custom definition of CoVs for grouped countries